# Signisa — Phase 1 training (GPU, internet ON)

Attach the `kaggle_prep` notebook's output as input. Edit only the CONFIG cell.
Trains the cnn_transformer embedder (CE or ArcFace), then runs the signer-independent
verification evaluation and writes metrics_report.md + curriculum_db_trained.json + model.pt.

In [ ]:
# CONFIG — the only cell to edit
LOSS = "arcface"       # "ce" | "arcface"
EPOCHS = 200
LR = 1e-3
BATCH_SIZE = 256
PATIENCE = 30          # early-stopping patience on val top-1
TENSORS_DIR = "/kaggle/input/kaggle-prep/tensors"

In [ ]:
!git clone -q https://github.com/dayan-battulga/signisa.git /kaggle/working/signisa-repo
%pip install -q /kaggle/working/signisa-repo

In [ ]:
import pandas as pd
import torch

from signisa.config import Config
from signisa.data import ShardDataset
from signisa.eval import held_out_participants, run_evaluation
from signisa.train import train_model

index_head = pd.read_csv(f"{TENSORS_DIR}/index.csv", nrows=1)
LANDMARK_VERSION = (index_head.landmark_version.iloc[0]
                    if "landmark_version" in index_head.columns else "v1")
print("tensors landmark_version:", LANDMARK_VERSION)
cfg = Config(loss=LOSS, epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE, patience=PATIENCE,
             landmark_version=LANDMARK_VERSION)
device = "cuda" if torch.cuda.is_available() else "cpu"

index = pd.read_csv(f"{TENSORS_DIR}/index.csv")
val_pids = held_out_participants(index.participant_id, cfg.n_val_participants, cfg.seed)
train_pids = [p for p in sorted(index.participant_id.unique()) if p not in val_pids]
print("val participants:", val_pids)

train_ds = ShardDataset(TENSORS_DIR, augment=True, participants=train_pids)
val_ds = ShardDataset(TENSORS_DIR, participants=val_pids)
model, history = train_model(cfg, train_ds, val_ds, device=device)
best_epoch = int(max(range(len(history["val_top1"])), key=history["val_top1"].__getitem__))
tail = history["val_top1"][-20:]
print(f"best val top-1 {max(history['val_top1']):.1%} at epoch {best_epoch + 1} "
      f"of {len(history['val_top1'])} run")
print("val top-1 tail (last 20):", " ".join(f"{v:.3f}" for v in tail))

In [ ]:
REPO = "/kaggle/working/signisa-repo"
metrics = run_evaluation(
    model, TENSORS_DIR, f"{REPO}/data/meta/curriculum_db.json",
    f"{REPO}/data/meta/training_labels.json", cfg, "/kaggle/working",
    device=device, val_participants=val_pids)
from signisa.models import save_checkpoint
save_checkpoint(model, "/kaggle/working/model.pt")

# append training curve summary so the report shows whether EPOCHS was enough
with open("/kaggle/working/metrics_report.md", "a") as f:
    f.write(f"\n## Training\n\n- Best val top-1 {max(history['val_top1']):.1%} at epoch "
            f"{best_epoch + 1}; ran {len(history['val_top1'])}/{EPOCHS} epochs "
            f"(early-stopped: {'yes' if len(history['val_top1']) < EPOCHS else 'no'}).\n"
            f"- val top-1 tail (last 20): {' '.join(f'{v:.3f}' for v in tail)}\n"
            f"- Still climbing at the end suggests raising EPOCHS; a long flat tail "
            f"suggests it converged.\n")

print(f"TAR@FAR5: {metrics['tar_at_far']:.1%} | closed-set top-1: {metrics['top1_closed_set']:.1%}")
collapsed = [c["members"] for c in metrics["clusters"] if c["collapsed"]]
print("collapsed clusters (kill criterion):", collapsed or "none")